# 02 - Simulador, feedback y logs

Este notebook muestra el flujo offline completo: cierre, correccion identity, escritura de feedback JSONL y logs de debug.

El objetivo es que alguien que entra al repo pueda ver que el modulo no es solo una interfaz: produce artefactos auditables y mide latencias.

## 1. Setup y rutas de salida

Los archivos generados se escriben dentro de `realtime/outputs/`, que esta ignorado por Git.

In [1]:
from pathlib import Path
import csv
import json
import sys
from collections import Counter
from IPython.display import Markdown, display

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "realtime" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def _fmt(value):
    if isinstance(value, float):
        return f"{value:.4f}"
    if isinstance(value, (list, tuple)):
        return ", ".join(str(item) for item in value) or "-"
    text = str(value)
    return text.replace("\n", " ").replace("|", "\\|")


def show_table(rows, columns):
    if not rows:
        display(Markdown("_Sin filas para mostrar._"))
        return
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(_fmt(row.get(col, "")) for col in columns) + " |")
    display(Markdown("\n".join([header, sep, *body])))


def show_json(payload):
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

print(f"Repo detectado: {ROOT.name}")
from realtime.src.simular_flujo import DEMO_TEXTS, run_simulation
from realtime.src.provider_factory import make_closure_provider, make_correction_provider

out_dir = ROOT / "realtime" / "outputs" / "notebooks" / "02_simulador"
out_dir.mkdir(parents=True, exist_ok=True)
log_path = out_dir / "llm_logs.jsonl"
feedback_path = out_dir / "feedback.jsonl"
for path in (log_path, feedback_path):
    if path.exists():
        path.unlink()

print("Log JSONL:", log_path.relative_to(ROOT))
print("Feedback JSONL:", feedback_path.relative_to(ROOT))
print("Textos demo:", len(DEMO_TEXTS))

Repo detectado: labios-argentos
Log JSONL: realtime\outputs\notebooks\02_simulador\llm_logs.jsonl
Feedback JSONL: realtime\outputs\notebooks\02_simulador\feedback.jsonl
Textos demo: 8


## 2. Textos usados en la simulacion

Incluyen texto vacio, conectores colgantes, repeticiones, frases largas y frases con puntuacion.

In [2]:
show_table(
    [{"idx": idx, "texto": text or "<vacio>"} for idx, text in enumerate(DEMO_TEXTS, start=1)],
    ["idx", "texto"],
)

| idx | texto |
| --- | --- |
| 1 | <vacio> |
| 2 | yo creo que |
| 3 | hola como estas todo bien gracias por venir hoy |
| 4 | bueno bueno bueno bueno |
| 5 | me parece que |
| 6 | ayer fuimos a la cancha y estuvo buenisimo. |
| 7 | no se si |
| 8 | vos tenes razon che gracias por avisarme |

## 3. Corrida del flujo offline

Se usa `heuristic` para cierre e `identity` para correccion. No se invoca ningun backend LLM externo.

In [3]:
summary = run_simulation(
    DEMO_TEXTS,
    closure_provider_name="heuristic",
    correction_provider_name="identity",
    log_path=log_path,
    feedback_path=feedback_path,
)
show_json(summary)

{
  "actions": {
    "commit": 2,
    "low_confidence": 2,
    "wait": 4
  },
  "count": 8,
  "fallbacks": 0,
  "latency_ms": {
    "closure": {
      "p50": 0.1438,
      "p95": 0.5096
    },
    "correction": {
      "p50": 0.0434,
      "p95": 0.0475
    },
    "logging": {
      "p50": 1.6336,
      "p95": 4.08
    },
    "validation": {
      "p50": 0.0347,
      "p95": 0.0556
    }
  },
  "providers": {
    "closure": "heuristic_closure",
    "correction": "identity_correction"
  }
}


## 4. Resumen operativo

Este resumen es lo que mirariamos en una corrida rapida: cuantas acciones salieron, latencias y fallbacks.

In [4]:
lat = summary["latency_ms"]
summary_rows = [
    {"metrica": "ejemplos", "valor": summary["count"]},
    {"metrica": "provider_cierre", "valor": summary["providers"]["closure"]},
    {"metrica": "provider_correccion", "valor": summary["providers"]["correction"]},
    {"metrica": "commits", "valor": summary["actions"]["commit"]},
    {"metrica": "waits", "valor": summary["actions"]["wait"]},
    {"metrica": "low_confidence", "valor": summary["actions"]["low_confidence"]},
    {"metrica": "closure_p50_ms", "valor": lat["closure"]["p50"]},
    {"metrica": "closure_p95_ms", "valor": lat["closure"]["p95"]},
    {"metrica": "correction_p50_ms", "valor": lat["correction"]["p50"]},
    {"metrica": "logging_p95_ms", "valor": lat["logging"]["p95"]},
    {"metrica": "fallbacks", "valor": summary["fallbacks"]},
]
show_table(summary_rows, ["metrica", "valor"])

| metrica | valor |
| --- | --- |
| ejemplos | 8 |
| provider_cierre | heuristic_closure |
| provider_correccion | identity_correction |
| commits | 2 |
| waits | 4 |
| low_confidence | 2 |
| closure_p50_ms | 0.1438 |
| closure_p95_ms | 0.5096 |
| correction_p50_ms | 0.0434 |
| logging_p95_ms | 4.0800 |
| fallbacks | 0 |

## 5. Feedback generado

Cada commit genera un evento pendiente de revision humana. Esa es la base del futuro feedback loop.

In [5]:
def read_jsonl(path):
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

feedback_rows_raw = read_jsonl(feedback_path)
feedback_rows = []
for event in feedback_rows_raw:
    feedback_rows.append({
        "segment_id": event["segment_id"],
        "raw_vsr_text": event["raw_vsr_text"],
        "committed_text": event["committed_text"],
        "corrected_text": event["corrected_text"],
        "review_status": event["review_status"],
        "latency_ms": round(event["latency_ms"], 4),
    })
show_table(feedback_rows, ["segment_id", "raw_vsr_text", "committed_text", "corrected_text", "review_status", "latency_ms"])
print("Eventos de feedback:", len(feedback_rows_raw))

| segment_id | raw_vsr_text | committed_text | corrected_text | review_status | latency_ms |
| --- | --- | --- | --- | --- | --- |
| sim_0003 | hola como estas todo bien gracias por venir hoy | hola como estas todo bien gracias por venir hoy | hola como estas todo bien gracias por venir hoy | pending | 0.2482 |
| sim_0006 | ayer fuimos a la cancha y estuvo buenisimo. | ayer fuimos a la cancha y estuvo buenisimo. | ayer fuimos a la cancha y estuvo buenisimo. | pending | 0.5490 |

Eventos de feedback: 2


## 6. Logs de provider

Los logs guardan entrada, salida, latencia, validacion y fallback. Sirven para debug y para armar datasets de evaluacion despues.

In [6]:
log_rows_raw = read_jsonl(log_path)
log_rows = []
for event in log_rows_raw[:6]:
    output = event.get("output", {})
    log_rows.append({
        "stage": event.get("stage"),
        "provider": event.get("provider"),
        "latency_ms": round(event.get("latency_ms", 0.0), 4),
        "valid": event.get("valid"),
        "fallback": event.get("fallback_used"),
        "action/cambio": output.get("action", output.get("changed", "")),
        "reason": output.get("reason", ""),
    })
show_table(log_rows, ["stage", "provider", "latency_ms", "valid", "fallback", "action/cambio", "reason"])
print("Eventos de log:", len(log_rows_raw))

| stage | provider | latency_ms | valid | fallback | action/cambio | reason |
| --- | --- | --- | --- | --- | --- | --- |
| closure | heuristic_closure | 0.1038 | True | False | low_confidence | texto_vacio |
| closure | heuristic_closure | 0.2025 | True | False | wait | conector_colgante |
| closure | heuristic_closure | 0.2007 | True | False | commit | contexto_suficiente_sin_conector_colgante |
| correction | identity_correction | 0.0475 | True | False | False |  |
| closure | heuristic_closure | 0.1488 | True | False | low_confidence | texto_repetitivo |
| closure | heuristic_closure | 0.1388 | True | False | wait | conector_colgante |

Eventos de log: 10


## 7. Provider LLM opcional

El modulo ya tiene una interfaz reemplazable para un provider LLM/Ollama, pero este notebook no lo requiere. La corrida presentable de etapa 1 queda cerrada con la baseline local.

In [7]:
providers = {
    "closure_default": make_closure_provider("heuristic").name,
    "correction_default": make_correction_provider("identity").name,
    "ollama_disponible_como_opcion": "ollama" in ["heuristic", "ollama"],
    "requiere_servicio_externo_para_mvp": False,
}
show_json(providers)

{
  "closure_default": "heuristic_closure",
  "correction_default": "identity_correction",
  "ollama_disponible_como_opcion": true,
  "requiere_servicio_externo_para_mvp": false
}


## 8. Lectura final

Resultado de este notebook:

- El flujo completo corre offline.
- Se generan feedback y logs JSONL.
- Las latencias quedan medidas por etapa.
- El provider LLM queda preparado, pero no bloquea el MVP.